# Treino LSTM - CHiME6 (ASR Front-End)

Objetivo: treinar um classificador temporal (fala vs ruido/silencio) para servir de pre-processador ao ASR.

Regras deste notebook:
- Sem shuffle temporal de linhas
- Busca de hiperparametros com amostragem temporal contigua
- `dev` para treino/otimizacao
- `eval` dividido temporalmente em duas metades: `evaluation` e `test`

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import optuna

from chime_lstm_data import (
    load_feature_csv,
    detect_feature_columns,
    temporal_split_by_group,
    split_eval_half_timeline,
    contiguous_subsample_by_group,
    build_sequences,
)
from lstm_model import (
    LSTMClassifier,
    train_sequence_model,
    infer_records, evaluate_model,
    compute_class_weights,
    save_model,
    compute_metrics,
)

In [ ]:
BASE_DIR = Path('.')
DEV_CSV = BASE_DIR / 'chime6_kinect_dev.csv'
EVAL_CSV = BASE_DIR / 'chime6_kinect_eval.csv'

ARTIFACTS_DIR = BASE_DIR / 'results' / 'lstm_asr'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 150

# Frações temporais
DEV_TRAIN_RATIO = 0.85
OPTUNA_DEV_FRAC = 0.30

# Busca de hiperparametros
N_TRIALS = 15
OPTUNA_EPOCHS = 20
FINAL_EPOCHS = 40
OBJECTIVE_METRIC = 'f1'

# Early stopping
PATIENCE = 10
MIN_DELTA = 1e-3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Carregar base e criar splits temporais

In [ ]:
df_dev = load_feature_csv(str(DEV_CSV))
df_eval_all = load_feature_csv(str(EVAL_CSV))

feature_cols = detect_feature_columns(df_dev)
print('n_features:', len(feature_cols))

df_train_raw, df_val_raw = temporal_split_by_group(df_dev, train_ratio=DEV_TRAIN_RATIO)
df_eval_raw, df_test_raw = split_eval_half_timeline(df_eval_all)

print('Rows train/val/eval/test:', len(df_train_raw), len(df_val_raw), len(df_eval_raw), len(df_test_raw))
print('Class ratio train:', df_train_raw['label'].mean().round(4))
print('Class ratio val  :', df_val_raw['label'].mean().round(4))
print('Class ratio eval :', df_eval_raw['label'].mean().round(4))
print('Class ratio test :', df_test_raw['label'].mean().round(4))

## 2. Escalonamento

> As features do CSV do pipeline CHiME6 já vêm normalizadas (z-score por feature).
> Portanto, não aplicamos `StandardScaler` adicional para evitar dupla normalização.

In [ ]:
# Features já normalizadas no pipeline de extração CHiME6.
# Mantemos os dataframes como estão (sem scaling adicional).
df_train = df_train_raw.copy()
df_val = df_val_raw.copy()
df_eval = df_eval_raw.copy()
df_test = df_test_raw.copy()

normalization_info = {
    'feature_cols': feature_cols,
    'seq_len': SEQ_LEN,
    'extra_scaling_applied': False,
    'note': 'Input CSV already normalized in chime6_pds_pipeline.ipynb',
}
with open(ARTIFACTS_DIR / 'normalization_config.json', 'w', encoding='utf-8') as f:
    json.dump(normalization_info, f, indent=2)

print('Sem escalonamento adicional. Config salvo em', ARTIFACTS_DIR / 'normalization_config.json')

## 3. Busca de hiperparametros com amostragem temporal contigua

In [ ]:
df_train_opt = contiguous_subsample_by_group(df_train_raw, frac=OPTUNA_DEV_FRAC, from_start=True)
df_val_opt = contiguous_subsample_by_group(df_val_raw, frac=OPTUNA_DEV_FRAC, from_start=True)

print('Optuna rows train/val:', len(df_train_opt), len(df_val_opt))

In [ ]:
def objective(trial):
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 96, 128, 192])
    num_layers = trial.suggest_categorical('num_layers', [1, 2, 3])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])
    stride = trial.suggest_categorical('stride', [50, 75, 100, 150])

    train_pack = build_sequences(df_train_opt, feature_cols, seq_len=SEQ_LEN, stride=stride, label_mode='last')
    val_pack = build_sequences(df_val_opt, feature_cols, seq_len=SEQ_LEN, stride=stride, label_mode='last')

    if len(train_pack.y) == 0 or len(val_pack.y) == 0:
        return 0.0

    model = LSTMClassifier(
        input_dim=len(feature_cols),
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout_rate=dropout_rate,
        output_dim=1,
    )

    class_weights = compute_class_weights(train_pack.y)
    train_out = train_sequence_model(
        model=model,
        X_train=train_pack.X,
        y_train=train_pack.y,
        X_val=val_pack.X,
        y_val=val_pack.y,
        epochs=OPTUNA_EPOCHS,
        batch_size=batch_size,
        lr=lr,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        device=DEVICE,
        class_weights=class_weights,
        loss_name='bce_with_logits',
    )
    model = train_out['model']
    metrics = evaluate_model(model, val_pack.X, val_pack.y, batch_size=batch_size, split_name='val', device=DEVICE)
    best_value = metrics[OBJECTIVE_METRIC]
    return float(best_value)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'Best {OBJECTIVE_METRIC.upper()}:', study.best_value)
print('Best params:', study.best_params)

## 4. Treino final com melhores hiperparametros

In [ ]:
best = study.best_params
best_stride = int(best['stride'])

pack_train = build_sequences(df_train_raw, feature_cols, seq_len=SEQ_LEN, stride=best_stride, label_mode='last')
pack_val = build_sequences(df_val_raw, feature_cols, seq_len=SEQ_LEN, stride=best_stride, label_mode='last')
pack_eval = build_sequences(df_eval_raw, feature_cols, seq_len=SEQ_LEN, stride=best_stride, label_mode='last')
pack_test = build_sequences(df_test_raw, feature_cols, seq_len=SEQ_LEN, stride=best_stride, label_mode='last')

print('Records train/val/eval/test:', len(pack_train.y), len(pack_val.y), len(pack_eval.y), len(pack_test.y))

final_model = LSTMClassifier(
    input_dim=len(feature_cols),
    hidden_dim=int(best['hidden_dim']),
    num_layers=int(best['num_layers']),
    dropout_rate=float(best['dropout_rate']),
    output_dim=1,
)

final_out = train_sequence_model(
    model=final_model,
    X_train=pack_train.X,
    y_train=pack_train.y,
    X_val=pack_val.X,
    y_val=pack_val.y,
    epochs=FINAL_EPOCHS,
    batch_size=int(best['batch_size']),
    lr=float(best['lr']),
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    device=DEVICE,
    class_weights=compute_class_weights(pack_train.y),
    loss_name='bce_with_logits',
)
trained_model = final_out['model']
print('Best val_loss (final):', final_out['best_val_loss'])

## 5. Métricas gerais de classificação (train/val/eval/test)

In [ ]:
def eval_pack(name, pack):
    preds, scores = infer_records(trained_model, pack.X, batch_size=256, device=DEVICE)
    m = compute_metrics(pack.y, preds, scores)
    m['split'] = name
    m['n_records'] = int(len(pack.y))
    return m

metrics_train = eval_pack('train', pack_train)
metrics_val = eval_pack('val', pack_val)
metrics_eval = eval_pack('evaluation', pack_eval)
metrics_test = eval_pack('test', pack_test)

metrics_all = [metrics_train, metrics_val, metrics_eval, metrics_test]
df_metrics = pd.DataFrame(metrics_all)
display(df_metrics[['split', 'n_records', 'accuracy', 'precision', 'recall', 'f1', 'auroc']].round(4))

## 6. Exportar modelo e artefatos

In [ ]:
model_path = ARTIFACTS_DIR / 'lstm_asr_best.pt'
save_model(trained_model, str(model_path))

export_payload = {
    'best_params': {k: (float(v) if isinstance(v, (np.floating,)) else int(v) if isinstance(v, (np.integer,)) else v) for k, v in best.items()},
    'seq_len': SEQ_LEN,
    'feature_count': len(feature_cols),
    'metrics': metrics_all,
}

with open(ARTIFACTS_DIR / 'metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(export_payload, f, indent=2)

with open(ARTIFACTS_DIR / 'best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best, f, indent=2)

df_metrics.to_csv(ARTIFACTS_DIR / 'metrics_summary.csv', index=False)

print('Modelo salvo em:', model_path)
print('Métricas salvas em:', ARTIFACTS_DIR / 'metrics_summary.json')